In [4]:
import rasterio
from rasterio.warp import calculate_default_transform, reproject, Resampling
import pyproj
from pyproj import CRS


In [6]:
try:
    crs = CRS.from_epsg(54009)  # or try 54030 for Mollweide
    print(crs)
except Exception as e:
    print(f"Error: {e}")

Error: Invalid projection: epsg:54009: (Internal Proj Error: proj_create: crs not found)


In [2]:
file = '/Users/bz294/Downloads/wc2.1_5m_bioc_ACCESS-CM2_ssp370_2081-2100.tif'

In [3]:
input_file = '/Users/bz294/Downloads/wc2.1_5m_bioc_ACCESS-CM2_ssp370_2081-2100.tif'
output_file = '/Users/bz294/Downloads/reprojected_bioc.tif'

# Specify the band you want to extract (e.g., band 1)
band_to_extract = 1  # Change this value to the band you want (1-19)

# Open the input raster file
with rasterio.open(input_file) as src:
    # Define the target CRS (EPSG:54009)
    target_crs = 'EPSG:54009'
    
    # Calculate the transform and new dimensions for the target CRS
    transform, width, height = calculate_default_transform(
        src.crs, target_crs, src.width, src.height, *src.bounds
    )
    
    # Set the metadata for the output file
    kwargs = src.meta.copy()
    kwargs.update({
        'crs': target_crs,
        'transform': transform,
        'width': width,
        'height': height,
        'count': 1  # Only one band in the output
    })
    
    # Read the specific band you want to extract
    band_data = src.read(band_to_extract)

    # Reproject the extracted band
    with rasterio.open(output_file, 'w', **kwargs) as dst:
        reproject(
            source=band_data,
            destination=rasterio.band(dst, 1),
            src_transform=src.transform,
            src_crs=src.crs,
            dst_transform=transform,
            dst_crs=target_crs,
            resampling=Resampling.nearest
        )

print(f"Reprojected single band raster saved to: {output_file}")

CRSError: The EPSG code is unknown. PROJ: proj_create_from_database: crs not found